# Generate Submission


In [ ]:
# If needed in Colab, uncomment:
!pip -q install "transformers>=4.47.0" peft bitsandbytes accelerate pillow pandas numpy

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!cp "/content/drive/MyDrive/DL_final/data.zip" /content/data.zip
!unzip -q /content/data.zip -d /content/

In [ ]:
import json
from pathlib import Path
from typing import List

import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn.functional as F

from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText,
    BitsAndBytesConfig,
)
from peft import PeftModel


In [ ]:
MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"
DATA_DIR = Path("/content")
OUT_DIR = Path("/content/drive/MyDrive/DL_final/artifacts/lora_full_redo")
OUT_DIR.mkdir(parents=True, exist_ok=True)

best_checkpoint_path = OUT_DIR / "best_checkpoint.txt"

if best_checkpoint_path.exists():
    ADAPTER_DIR = Path(best_checkpoint_path.read_text().strip())
else:
    BEST_RUN_NAME = "smolvlm_qlora_allLayers_run1_q_hint_lecture_lr0.0001"
    ADAPTER_DIR = OUT_DIR / BEST_RUN_NAME / "best_adapter"

SUBMISSION_PATH = OUT_DIR / "submission.csv"

print("Adapter dir:", ADAPTER_DIR)

In [ ]:
summary_path = OUT_DIR / "experiment_summary.csv"
cfg_path = ADAPTER_DIR / "run_config.json"

selected_run_name = None

if cfg_path.exists():
    print("run_config.json for selected checkpoint:")
    cfg = json.loads(cfg_path.read_text())
    print(json.dumps(cfg, indent=2))

    selected_run_name = cfg.get("run_name", None)
else:
    print(f"No run_config.json found at: {cfg_path}")

if summary_path.exists():
    s = pd.read_csv(summary_path)

    print("experiment_summary.csv row for selected checkpoint/run:")

    if "best_checkpoint_dir" in s.columns:
        selected = s[
            s["best_checkpoint_dir"].astype(str) == str(ADAPTER_DIR)
        ]
    elif selected_run_name is not None and "run_name" in s.columns:
        selected = s[s["run_name"] == selected_run_name]
    else:
        selected = pd.DataFrame()

    display(selected)

In [ ]:
CHOICE_LETTERS = "ABCDEFG"

def resolve_image_path(data_dir: Path, rel_path: str) -> Path:
    p1 = data_dir / rel_path
    if p1.exists():
        return p1
    p2 = data_dir / "images" / rel_path
    if p2.exists():
        return p2
    p3 = data_dir / "images" / "images" / rel_path.replace("images/", "", 1)
    if p3.exists():
        return p3
    raise FileNotFoundError(f"Could not resolve image path: {rel_path}")

def build_user_text(row: pd.Series, choices: List[str], context_mode: str = "q_hint_lecture") -> str:
    context_parts = []

    hint = row.get("hint", "")
    lecture = row.get("lecture", "")

    if context_mode in ["q_hint", "q_hint_lecture"]:
        if pd.notna(hint) and str(hint).strip():
            context_parts.append(f"Hint: {str(hint).strip()}")

    if context_mode == "q_hint_lecture":
        if pd.notna(lecture) and str(lecture).strip():
            context_parts.append(f"Lecture: {str(lecture).strip()}")

    choice_lines = [f"{CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices)]

    parts = [
        "Answer the science multiple-choice question using the image.",
        "Respond with exactly one capital letter.",
    ]
    if context_parts:
        parts.append("\n".join(context_parts))
    parts.append(f"Question: {row['question']}")
    parts.append("Choices:\n" + "\n".join(choice_lines))

    return "\n\n".join(parts)

def build_messages(row: pd.Series, choices: List[str], context_mode: str = "q_hint_lecture"):
    text = build_user_text(row, choices, context_mode=context_mode)
    return [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": text},
            ],
        }
    ]

In [ ]:
test_df = pd.read_csv(DATA_DIR / "test.csv")
test_df["choices"] = test_df["choices"].apply(json.loads)
print("Test rows:", len(test_df))
display(test_df.head(2))

In [ ]:
processor_source = ADAPTER_DIR if ADAPTER_DIR.exists() else MODEL_ID
processor = AutoProcessor.from_pretrained(processor_source)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
)

base = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto" if torch.cuda.is_available() else None,
    low_cpu_mem_usage=True,
)

model = PeftModel.from_pretrained(base, ADAPTER_DIR)
model.eval()

model_device = next(model.parameters()).device
print("Model device:", model_device)

In [ ]:
def make_letter_token_id_map(tokenizer):
    token_map = {}
    for ch in CHOICE_LETTERS:
        token_ids = []
        for variant in [ch, " " + ch]:
            ids = tokenizer(variant, add_special_tokens=False).input_ids
            if len(ids) == 1:
                token_ids.append(ids[0])
        token_ids = sorted(set(token_ids))
        if not token_ids:
            raise ValueError(f"No single-token form found for {ch}")
        token_map[ch] = token_ids
    return token_map

letter_token_map = make_letter_token_id_map(processor.tokenizer)

context_mode = "q_hint_lecture"
cfg_path = ADAPTER_DIR / "run_config.json"
if cfg_path.exists():
    cfg = json.loads(cfg_path.read_text())
    context_mode = cfg.get("context_mode", context_mode)

print("Using context_mode:", context_mode)
print("Letter token map:", letter_token_map)

In [ ]:
MAX_TEXT_LENGTH = 1536
BATCH_SIZE = 16

@torch.no_grad()
def predict_batch(rows: pd.DataFrame):
    images = []
    texts = []
    num_choices = []

    for _, row in rows.iterrows():
        img = Image.open(resolve_image_path(DATA_DIR, row["image_path"])).convert("RGB")
        choices = row["choices"]
        messages = build_messages(row, choices, context_mode=context_mode)
        rendered = processor.apply_chat_template(messages, add_generation_prompt=True)
        images.append(img)
        texts.append(rendered)
        num_choices.append(len(choices))

    proc = processor(
        text=texts,
        images=images,
        return_tensors="pt",
        padding=True,
    )
    proc = {k: (v.to(model_device) if torch.is_tensor(v) else v) for k, v in proc.items()}

    out = model(**proc)
    logits = out.logits
    attn = proc["attention_mask"]
    last_idx = attn.sum(dim=1) - 1
    next_logits = logits[torch.arange(logits.size(0), device=model_device), last_idx, :]

    log_probs = F.log_softmax(next_logits, dim=-1)

    choice_scores = torch.full((logits.size(0), len(CHOICE_LETTERS)), -1e9, device=model_device)
    for j, ch in enumerate(CHOICE_LETTERS):
        tok_ids = torch.tensor(letter_token_map[ch], device=model_device)
        choice_scores[:, j] = torch.logsumexp(log_probs[:, tok_ids], dim=1)

    num_choices_t = torch.tensor(num_choices, device=model_device)
    invalid_mask = torch.arange(len(CHOICE_LETTERS), device=model_device).unsqueeze(0) >= num_choices_t.unsqueeze(1)
    choice_scores = choice_scores.masked_fill(invalid_mask, -1e9)

    preds = torch.argmax(choice_scores, dim=1).tolist()
    return preds

In [ ]:
preds = []
for i in tqdm(range(0, len(test_df), BATCH_SIZE), desc="Generating submission", total=(len(test_df) + BATCH_SIZE - 1) // BATCH_SIZE):
    batch_df = test_df.iloc[i:i + BATCH_SIZE]
    preds.extend(predict_batch(batch_df))

sub = pd.DataFrame({"id": test_df["id"].values, "answer": preds})

assert list(sub.columns) == ["id", "answer"]
assert len(sub) == len(test_df)
for p, n in zip(sub["answer"].tolist(), test_df["num_choices"].tolist()):
    assert 0 <= int(p) < int(n)

sub.to_csv(SUBMISSION_PATH, index=False)
print("Saved:", SUBMISSION_PATH)
display(sub.head(10))

In [ ]:
sub.to_csv(OUT_DIR / "submission1.csv", index=False)
print("Saved:", OUT_DIR / "submission1.csv")
display(sub.head(10))

In [ ]:
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")
print("Sample columns:", sample.columns.tolist())
print("Your columns:", sub.columns.tolist())
print("Rows sample vs your submission:", len(sample), len(sub))